In [13]:
import pandas as pd
import numpy as np
import pickle

pd.set_option("display.max_columns", None)

### Loading Master Provider Data

In [14]:
df_provider = pd.read_csv("../Train_Test_Data/Train_Master_Providers.csv")
df_provider['PotentialFraud'] = df_provider['PotentialFraud'].map({"No":0,"Yes":1})
print(df_provider.shape)
print("Unique Provider Ids: ", df_provider.Provider.nunique())
df_provider.head()

(5410, 2)
Unique Provider Ids:  5410


,Provider,PotentialFraud
0,PRV51001,0
1,PRV51003,1
2,PRV51004,0
3,PRV51005,1
4,PRV51007,0


### Loading Benefitiory Data

In [15]:
df_benf = pd.read_csv("../Train_Test_Data/Train_Beneficiarydata.csv")
df_benf.drop(columns = ['DOB','DOD','Gender','Race','State','County'], inplace = True)
print(df_benf.shape)
df_benf.head()

(138556, 19)


,BeneID,RenalDiseaseIndicator,NoOfMonths_PartACov,NoOfMonths_PartBCov,ChronicCond_Alzheimer,ChronicCond_Heartfailure,ChronicCond_KidneyDisease,ChronicCond_Cancer,ChronicCond_ObstrPulmonary,ChronicCond_Depression,ChronicCond_Diabetes,ChronicCond_IschemicHeart,ChronicCond_Osteoporasis,ChronicCond_rheumatoidarthritis,ChronicCond_stroke,IPAnnualReimbursementAmt,IPAnnualDeductibleAmt,OPAnnualReimbursementAmt,OPAnnualDeductibleAmt
0,BENE11001,0,12,12,1,2,1,2,2,1,1,1,2,1,1,36000,3204,60,70
1,BENE11002,0,12,12,2,2,2,2,2,2,2,2,2,2,2,0,0,30,50
2,BENE11003,0,12,12,1,2,2,2,2,2,2,1,2,2,2,0,0,90,40
3,BENE11004,0,12,12,1,1,2,2,2,2,1,1,1,1,2,0,0,1810,760
4,BENE11005,0,12,12,2,2,2,2,1,2,1,2,2,2,2,0,0,1790,1200


In [16]:
mapping_cols = ['ChronicCond_Alzheimer', 'ChronicCond_Heartfailure',
       'ChronicCond_KidneyDisease', 'ChronicCond_Cancer',
       'ChronicCond_ObstrPulmonary', 'ChronicCond_Depression',
       'ChronicCond_Diabetes', 'ChronicCond_IschemicHeart',
       'ChronicCond_Osteoporasis', 'ChronicCond_rheumatoidarthritis',
       'ChronicCond_stroke']



for c in mapping_cols:
    df_benf[c] = df_benf[c].map({1:0,2:1})

df_benf['RenalDiseaseIndicator'] = df_benf['RenalDiseaseIndicator'].map({'0':0,'Y':1})

print(df_benf.shape)
df_benf.head()

(138556, 19)


,BeneID,RenalDiseaseIndicator,NoOfMonths_PartACov,NoOfMonths_PartBCov,ChronicCond_Alzheimer,ChronicCond_Heartfailure,ChronicCond_KidneyDisease,ChronicCond_Cancer,ChronicCond_ObstrPulmonary,ChronicCond_Depression,ChronicCond_Diabetes,ChronicCond_IschemicHeart,ChronicCond_Osteoporasis,ChronicCond_rheumatoidarthritis,ChronicCond_stroke,IPAnnualReimbursementAmt,IPAnnualDeductibleAmt,OPAnnualReimbursementAmt,OPAnnualDeductibleAmt
0,BENE11001,0,12,12,0,1,0,1,1,0,0,0,1,0,0,36000,3204,60,70
1,BENE11002,0,12,12,1,1,1,1,1,1,1,1,1,1,1,0,0,30,50
2,BENE11003,0,12,12,0,1,1,1,1,1,1,0,1,1,1,0,0,90,40
3,BENE11004,0,12,12,0,0,1,1,1,1,0,0,0,0,1,0,0,1810,760
4,BENE11005,0,12,12,1,1,1,1,0,1,0,1,1,1,1,0,0,1790,1200


### Loading IP Data

In [17]:
df_ip = pd.read_csv("../Train_Test_Data/Train_Inpatientdata.csv")
print(df_ip.shape)
df_ip.head()

(40474, 30)


,BeneID,ClaimID,ClaimStartDt,ClaimEndDt,Provider,InscClaimAmtReimbursed,AttendingPhysician,OperatingPhysician,OtherPhysician,AdmissionDt,ClmAdmitDiagnosisCode,DeductibleAmtPaid,DischargeDt,DiagnosisGroupCode,ClmDiagnosisCode_1,ClmDiagnosisCode_2,ClmDiagnosisCode_3,ClmDiagnosisCode_4,ClmDiagnosisCode_5,ClmDiagnosisCode_6,ClmDiagnosisCode_7,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6
0,BENE11001,CLM46614,2009-04-12,2009-04-18,PRV55912,26000,PHY390922,NaN,NaN,2009-04-12,7866,1068.0,2009-04-18,201,1970,4019,5853,7843,2768,71590,2724,19889,5849,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BENE11001,CLM66048,2009-08-31,2009-09-02,PRV55907,5000,PHY318495,PHY318495,NaN,2009-08-31,6186,1068.0,2009-09-02,750,6186,2948,56400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7092.0,NaN,NaN,NaN,NaN,NaN
2,BENE11001,CLM68358,2009-09-17,2009-09-20,PRV56046,5000,PHY372395,NaN,PHY324689,2009-09-17,29590,1068.0,2009-09-20,883,29623,30390,71690,34590,V1581,32723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BENE11011,CLM38412,2009-02-14,2009-02-22,PRV52405,5000,PHY369659,PHY392961,PHY349768,2009-02-14,431,1068.0,2009-02-22,067,43491,2762,7843,32723,V1041,4254,25062,40390,4019,NaN,331.0,NaN,NaN,NaN,NaN,NaN
4,BENE11014,CLM63689,2009-08-13,2009-08-30,PRV56614,10000,PHY379376,PHY398258,NaN,2009-08-13,78321,1068.0,2009-08-30,975,042,3051,34400,5856,42732,486,5119,29620,20300,NaN,3893.0,NaN,NaN,NaN,NaN,NaN


In [18]:
### Number of days stayed in Hospital
df_ip['AdmissionDt'] = pd.to_datetime(df_ip['AdmissionDt'], errors = 'coerce')
df_ip['DischargeDt'] = pd.to_datetime(df_ip['DischargeDt'], errors = 'coerce')

df_ip['IP_Number_of_Days_in_Hospital'] = (df_ip['DischargeDt'] - df_ip['AdmissionDt'] ).dt.days

### Number of Days taken to solve Claim
df_ip['ClaimStartDt'] = pd.to_datetime(df_ip['ClaimStartDt'], errors = 'coerce')
df_ip['ClaimEndDt'] = pd.to_datetime(df_ip['ClaimEndDt'], errors = 'coerce')

df_ip['IP_Claim_Days'] = (df_ip['ClaimEndDt'] - df_ip['ClaimStartDt'] ).dt.days


df_ip.drop(columns = ['AdmissionDt','DischargeDt','ClaimStartDt','ClaimEndDt','AttendingPhysician','OperatingPhysician','OtherPhysician'], inplace = True)

print(df_ip.shape)
df_ip.head()

(40474, 25)


,BeneID,ClaimID,Provider,InscClaimAmtReimbursed,ClmAdmitDiagnosisCode,DeductibleAmtPaid,DiagnosisGroupCode,ClmDiagnosisCode_1,ClmDiagnosisCode_2,ClmDiagnosisCode_3,ClmDiagnosisCode_4,ClmDiagnosisCode_5,ClmDiagnosisCode_6,ClmDiagnosisCode_7,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6,IP_Number_of_Days_in_Hospital,IP_Claim_Days
0,BENE11001,CLM46614,PRV55912,26000,7866,1068.0,201,1970,4019,5853,7843,2768,71590,2724,19889,5849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,6
1,BENE11001,CLM66048,PRV55907,5000,6186,1068.0,750,6186,2948,56400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7092.0,NaN,NaN,NaN,NaN,NaN,2,2
2,BENE11001,CLM68358,PRV56046,5000,29590,1068.0,883,29623,30390,71690,34590,V1581,32723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3
3,BENE11011,CLM38412,PRV52405,5000,431,1068.0,067,43491,2762,7843,32723,V1041,4254,25062,40390,4019,NaN,331.0,NaN,NaN,NaN,NaN,NaN,8,8
4,BENE11014,CLM63689,PRV56614,10000,78321,1068.0,975,042,3051,34400,5856,42732,486,5119,29620,20300,NaN,3893.0,NaN,NaN,NaN,NaN,NaN,17,17


In [19]:
### Number of Disease Diagnosied with
df_ip['IP_Unique_Disease_Count'] = df_ip[['ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3',
       'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6',
       'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9',
       'ClmDiagnosisCode_10']].count(axis =1)

### Number of Disease Diagnosied with
df_ip['IP_Unique_Treatment_Count'] = df_ip[['ClmProcedureCode_1', 'ClmProcedureCode_2',
       'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5',
       'ClmProcedureCode_6']].count(axis =1)

### Dropping Columns
df_ip.drop(columns = ['ClmAdmitDiagnosisCode','DiagnosisGroupCode','ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3',
       'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6',
       'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9',
       'ClmDiagnosisCode_10','ClmProcedureCode_1', 'ClmProcedureCode_2',
       'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5',
       'ClmProcedureCode_6'], inplace = True)
print(df_ip.shape)
df_ip.head()


(40474, 9)


,BeneID,ClaimID,Provider,InscClaimAmtReimbursed,DeductibleAmtPaid,IP_Number_of_Days_in_Hospital,IP_Claim_Days,IP_Unique_Disease_Count,IP_Unique_Treatment_Count
0,BENE11001,CLM46614,PRV55912,26000,1068.0,6,6,9,0
1,BENE11001,CLM66048,PRV55907,5000,1068.0,2,2,3,1
2,BENE11001,CLM68358,PRV56046,5000,1068.0,3,3,6,0
3,BENE11011,CLM38412,PRV52405,5000,1068.0,8,8,9,1
4,BENE11014,CLM63689,PRV56614,10000,1068.0,17,17,9,1


In [20]:

df_ip.rename(columns = {'ClaimID' : 'IP_ClaimID',
                        'InscClaimAmtReimbursed' : 'IP_InscClaimAmtReimbursed',
                        'DeductibleAmtPaid' : 'IP_DeductibleAmtPaid'}, inplace = True)

print(df_ip.shape)
df_ip.head()

(40474, 9)


,BeneID,IP_ClaimID,Provider,IP_InscClaimAmtReimbursed,IP_DeductibleAmtPaid,IP_Number_of_Days_in_Hospital,IP_Claim_Days,IP_Unique_Disease_Count,IP_Unique_Treatment_Count
0,BENE11001,CLM46614,PRV55912,26000,1068.0,6,6,9,0
1,BENE11001,CLM66048,PRV55907,5000,1068.0,2,2,3,1
2,BENE11001,CLM68358,PRV56046,5000,1068.0,3,3,6,0
3,BENE11011,CLM38412,PRV52405,5000,1068.0,8,8,9,1
4,BENE11014,CLM63689,PRV56614,10000,1068.0,17,17,9,1


### Loading OP Data

In [21]:
df_op = pd.read_csv("../Train_Test_Data/Train_Outpatientdata.csv")

df_op.drop(columns= ['AttendingPhysician','OperatingPhysician','OtherPhysician'],inplace = True)

print(df_op.shape)
df_op.head()

(517737, 24)


,BeneID,ClaimID,ClaimStartDt,ClaimEndDt,Provider,InscClaimAmtReimbursed,ClmDiagnosisCode_1,ClmDiagnosisCode_2,ClmDiagnosisCode_3,ClmDiagnosisCode_4,ClmDiagnosisCode_5,ClmDiagnosisCode_6,ClmDiagnosisCode_7,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6,DeductibleAmtPaid,ClmAdmitDiagnosisCode
0,BENE11002,CLM624349,10/11/2009,10/11/2009,PRV56011,30,78943,V5866,V1272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,56409
1,BENE11003,CLM189947,2/12/2009,2/12/2009,PRV57610,80,6115,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,79380
2,BENE11003,CLM438021,6/27/2009,6/27/2009,PRV57595,10,2723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
3,BENE11004,CLM121801,1/6/2009,1/6/2009,PRV56011,40,71988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
4,BENE11004,CLM150998,1/22/2009,1/22/2009,PRV56011,200,82382,30000,72887,4280,7197,V4577,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,71947


In [22]:
### Number of claim days
df_op['ClaimStartDt'] = pd.to_datetime(df_op['ClaimStartDt'], errors='coerce')
df_op['ClaimEndDt'] = pd.to_datetime(df_op['ClaimEndDt'], errors='coerce')

df_op['OP_Claim_Days'] = (df_op['ClaimEndDt'] - df_op['ClaimStartDt']).dt.days

df_op.drop(columns = ['ClaimStartDt','ClaimEndDt'], inplace = True)

print(df_op.shape)
df_op.tail()

(517737, 23)


,BeneID,ClaimID,Provider,InscClaimAmtReimbursed,ClmDiagnosisCode_1,ClmDiagnosisCode_2,ClmDiagnosisCode_3,ClmDiagnosisCode_4,ClmDiagnosisCode_5,ClmDiagnosisCode_6,ClmDiagnosisCode_7,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6,DeductibleAmtPaid,ClmAdmitDiagnosisCode,OP_Claim_Days
517732,BENE159198,CLM510792,PRV53699,800,2163,V4575,53190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
517733,BENE159198,CLM551294,PRV53702,400,07041,5781,25000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
517734,BENE159198,CLM596444,PRV53676,60,V570,78079,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
517735,BENE159198,CLM636992,PRV53689,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0
517736,BENE159198,CLM686139,PRV53689,80,78900,78609,4280,71946,3310,75311,2724,V103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,1


In [23]:
### Number of Disease Diagnosied with
df_op['OP_Unique_Disease_Count'] = df_op[['ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3',
       'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6',
       'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9',
       'ClmDiagnosisCode_10']].count(axis =1)

### Number of Disease Diagnosied with
df_op['OP_Unique_Treatment_Count'] = df_op[['ClmProcedureCode_1', 'ClmProcedureCode_2',
       'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5',
       'ClmProcedureCode_6']].count(axis =1)

### Dropping Columns
df_op.drop(columns = ['ClmAdmitDiagnosisCode','ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3',
       'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6',
       'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9',
       'ClmDiagnosisCode_10','ClmProcedureCode_1', 'ClmProcedureCode_2',
       'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5',
       'ClmProcedureCode_6'], inplace = True)
print(df_op.shape)
df_op.head()

(517737, 8)


,BeneID,ClaimID,Provider,InscClaimAmtReimbursed,DeductibleAmtPaid,OP_Claim_Days,OP_Unique_Disease_Count,OP_Unique_Treatment_Count
0,BENE11002,CLM624349,PRV56011,30,0,0,3,0
1,BENE11003,CLM189947,PRV57610,80,0,0,1,0
2,BENE11003,CLM438021,PRV57595,10,0,0,1,0
3,BENE11004,CLM121801,PRV56011,40,0,0,1,0
4,BENE11004,CLM150998,PRV56011,200,0,0,6,0


In [24]:
### Renaming Columns

df_op.rename(columns = {'ClaimID' : 'OP_ClaimID',
                        'InscClaimAmtReimbursed' : 'OP_InscClaimAmtReimbursed',
                        'DeductibleAmtPaid' : 'OP_DeductibleAmtPaid'}, inplace = True)

print(df_op.shape)
df_op.head()

(517737, 8)


,BeneID,OP_ClaimID,Provider,OP_InscClaimAmtReimbursed,OP_DeductibleAmtPaid,OP_Claim_Days,OP_Unique_Disease_Count,OP_Unique_Treatment_Count
0,BENE11002,CLM624349,PRV56011,30,0,0,3,0
1,BENE11003,CLM189947,PRV57610,80,0,0,1,0
2,BENE11003,CLM438021,PRV57595,10,0,0,1,0
3,BENE11004,CLM121801,PRV56011,40,0,0,1,0
4,BENE11004,CLM150998,PRV56011,200,0,0,6,0


### Merging IP Date with Master Provider Data

In [25]:
df_ip_provider = df_provider[['Provider']].merge(df_ip, on = 'Provider', how = 'left')

print(df_ip_provider.shape)
df_ip_provider.head(2)

(43792, 9)


,Provider,BeneID,IP_ClaimID,IP_InscClaimAmtReimbursed,IP_DeductibleAmtPaid,IP_Number_of_Days_in_Hospital,IP_Claim_Days,IP_Unique_Disease_Count,IP_Unique_Treatment_Count
0,PRV51001,BENE36012,CLM58316,36000.0,1068.0,4.0,4.0,8.0,0.0
1,PRV51001,BENE38773,CLM52334,12000.0,1068.0,2.0,2.0,6.0,0.0


In [26]:
df_ip_claim_benf = pd.DataFrame()
df_ip_claim_benf = df_ip_provider.groupby('Provider', as_index = False)[['IP_ClaimID','BeneID']].nunique()

df_ip_claim_benf.rename(columns = {"IP_ClaimID":"IP_Claim_Count", "BeneID" : "IP_Benf_Count"}, inplace = True)

print(df_ip_claim_benf.shape)
df_ip_claim_benf.head()

(5410, 3)


,Provider,IP_Claim_Count,IP_Benf_Count
0,PRV51001,5,5
1,PRV51003,62,53
2,PRV51004,0,0
3,PRV51005,0,0
4,PRV51007,3,3


In [27]:
df_ip_amount = pd.DataFrame()
df_ip_amount = df_ip_provider.groupby('Provider', as_index = False)[['IP_InscClaimAmtReimbursed','IP_DeductibleAmtPaid']].mean()

df_ip_amount.rename(columns = {'IP_InscClaimAmtReimbursed' : "Avg_IP_InscClaimAmtReimbursed","IP_DeductibleAmtPaid":"Avg_IP_DeductibleAmtPaid"}, inplace = True)

print(df_ip_amount.shape)
df_ip_amount.head()

(5410, 3)


,Provider,Avg_IP_InscClaimAmtReimbursed,Avg_IP_DeductibleAmtPaid
0,PRV51001,19400.000000,1068.0
1,PRV51003,9241.935484,1068.0
2,PRV51004,NaN,NaN
3,PRV51005,NaN,NaN
4,PRV51007,6333.333333,1068.0


In [28]:
df_ip_counts = df_ip_provider.groupby('Provider', as_index=False).agg({
    'IP_Number_of_Days_in_Hospital': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
    'IP_Claim_Days': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
    'IP_Unique_Disease_Count': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
    'IP_Unique_Treatment_Count': lambda x: x.mode().iloc[0] if not x.mode().empty else None
})

df_ip_counts.rename(columns = {
    'IP_Number_of_Days_in_Hospital': "Avg_IP_Number_of_Days_in_Hospital",
    'IP_Claim_Days': "Avg_IP_Number_of_Days_in_Hospital",
    'IP_Unique_Disease_Count': "Avg_IP_Number_of_Days_in_Hospital",
    'IP_Unique_Treatment_Count': "Avg_IP_Number_of_Days_in_Hospital" 
}, inplace= True)

print(df_ip_counts.shape)
df_ip_counts.head()

(5410, 5)


,Provider,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital
0,PRV51001,0.0,0.0,6.0,0.0
1,PRV51003,3.0,3.0,9.0,1.0
2,PRV51004,NaN,NaN,NaN,NaN
3,PRV51005,NaN,NaN,NaN,NaN
4,PRV51007,4.0,4.0,6.0,0.0


In [29]:
df_ip_final = df_ip_claim_benf.merge(df_ip_amount, on = 'Provider', how = 'left')
df_ip_final = df_ip_final.merge(df_ip_counts, on = 'Provider', how = 'left')

print(df_ip_final.shape)
df_ip_final.head()

(5410, 9)


,Provider,IP_Claim_Count,IP_Benf_Count,Avg_IP_InscClaimAmtReimbursed,Avg_IP_DeductibleAmtPaid,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital
0,PRV51001,5,5,19400.000000,1068.0,0.0,0.0,6.0,0.0
1,PRV51003,62,53,9241.935484,1068.0,3.0,3.0,9.0,1.0
2,PRV51004,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,PRV51005,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,PRV51007,3,3,6333.333333,1068.0,4.0,4.0,6.0,0.0


### Merging OP Data with Master Provider Data

In [30]:
df_op_provider=df_provider[['Provider']].merge(df_op,on='Provider',how='left')
print(df_op_provider.shape)
df_op_provider.head()

(518135, 8)


,Provider,BeneID,OP_ClaimID,OP_InscClaimAmtReimbursed,OP_DeductibleAmtPaid,OP_Claim_Days,OP_Unique_Disease_Count,OP_Unique_Treatment_Count
0,PRV51001,BENE11727,CLM733300,20.0,0.0,0.0,2.0,0.0
1,PRV51001,BENE24646,CLM372475,700.0,0.0,1.0,6.0,0.0
2,PRV51001,BENE31617,CLM748221,900.0,0.0,0.0,1.0,0.0
3,PRV51001,BENE32715,CLM272936,500.0,0.0,1.0,6.0,0.0
4,PRV51001,BENE49220,CLM452024,70.0,0.0,0.0,1.0,0.0


### 

In [ ]:
df_op_provider_claim_benf=df_op_provider.groupby('Provider')['BeneID','OP_ClaimID'].nunique()
df_op_provider_claim_benf.rename(columns={'BeneID':'OP_Benf_Count','OP_ClaimID':'OP_Claim_Count'},inplace=True)
df_op_provider_claim_benf.head()


C:\Users\MK PERUMALLA\AppData\Local\Temp\ipykernel_25488\221788602.py:1: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df_op_provider_claim_benf=df_op_provider.groupby('Provider')['BeneID','OP_ClaimID'].nunique()


,OP_Benf_Count,OP_Claim_Count
Provider,,
PRV51001,19,20
PRV51003,66,70
PRV51004,138,149
PRV51005,495,1165
PRV51007,56,69


In [34]:
df_op_provider_amt=df_op_provider.groupby('Provider')['OP_InscClaimAmtReimbursed','OP_DeductibleAmtPaid'].mean()
df_op_provider_amt.rename(columns={'OP_InscClaimAmtReimbursed':'Avg_OP_InscClaimAmtReimbursed','OP_DeductibleAmtPaid':'Avg_OP_DeductibleAmtPaid'},inplace=True)
df_op_provider_amt.head()


C:\Users\MK PERUMALLA\AppData\Local\Temp\ipykernel_25488\2651542188.py:1: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df_op_provider_amt=df_op_provider.groupby('Provider')['OP_InscClaimAmtReimbursed','OP_DeductibleAmtPaid'].mean()


,Avg_OP_InscClaimAmtReimbursed,Avg_OP_DeductibleAmtPaid
Provider,,
PRV51001,382.000000,0.000000
PRV51003,466.714286,1.000000
PRV51004,350.134228,2.080537
PRV51005,241.124464,3.175966
PRV51007,213.188406,0.869565


In [37]:
df_op_counts = df_op_provider.groupby('Provider', as_index=False).agg({
    'OP_Claim_Days': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
    'OP_Unique_Disease_Count': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
    'OP_Unique_Treatment_Count': lambda x: x.mode().iloc[0] if not x.mode().empty else None
})

df_op_counts.rename(columns = {
    'OP_Claim_Days': "Avg_OP_Number_of_Days_in_Hospital",
    'OP_Unique_Disease_Count': "Avg_OP_Number_of_Days_in_Hospital",
    'OP_Unique_Treatment_Count': "Avg_OP_Number_of_Days_in_Hospital" 
}, inplace= True)

print(df_op_counts.shape)
df_op_counts.head()

(5410, 4)


,Provider,Avg_OP_Number_of_Days_in_Hospital,Avg_OP_Number_of_Days_in_Hospital,Avg_OP_Number_of_Days_in_Hospital
0,PRV51001,0.0,1.0,0.0
1,PRV51003,0.0,1.0,0.0
2,PRV51004,0.0,1.0,0.0
3,PRV51005,0.0,1.0,0.0
4,PRV51007,0.0,1.0,0.0


In [38]:
df_op_final = df_op_provider_claim_benf.merge(df_op_provider_amt, on = 'Provider', how = 'left')
df_op_final = df_ip_final.merge(df_op_counts, on = 'Provider', how = 'left')

print(df_op_final.shape)
df_op_final.head()

(5410, 12)


,Provider,IP_Claim_Count,IP_Benf_Count,Avg_IP_InscClaimAmtReimbursed,Avg_IP_DeductibleAmtPaid,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_IP_Number_of_Days_in_Hospital,Avg_OP_Number_of_Days_in_Hospital,Avg_OP_Number_of_Days_in_Hospital,Avg_OP_Number_of_Days_in_Hospital
0,PRV51001,5,5,19400.000000,1068.0,0.0,0.0,6.0,0.0,0.0,1.0,0.0
1,PRV51003,62,53,9241.935484,1068.0,3.0,3.0,9.0,1.0,0.0,1.0,0.0
2,PRV51004,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0
3,PRV51005,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0
4,PRV51007,3,3,6333.333333,1068.0,4.0,4.0,6.0,0.0,0.0,1.0,0.0
